## Storing the vector in pinecone

In [ ]:
!pip install sentence-transformers pinecone-client python-dotenv requests

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving chunks.json to chunks (1).json


In [ ]:
pip install --upgrade pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 7.1 MB/s eta 0:00:00


In [ ]:
from pinecone import Pinecone

# Initialize Pinecone
pc = Pinecone(api_key="pcsk_VKN9t_D44vRHqfN6mmRtYYW85XgnaB2SPkx7k3vh2ZVY65tV2RDQgRMnZTpfnvRE3EHap")
index = pc.Index("credit-cards")

# Upsert vectors
vectors_to_upsert = []
for chunk, embedding in zip(chunks, embeddings):
    vectors_to_upsert.append({
        "id": chunk['chunk_id'],
        "values": embedding.tolist(),
        "metadata": {
            "card_name": chunk['card_name'],
            "page_number": chunk['page_number'],
            "chunk_text": chunk['chunk_text']
        }
    })

# Upload
index.upsert(vectors=vectors_to_upsert)
print(f"✓ Uploaded {len(vectors_to_upsert)} vectors")

✓ Uploaded 39 vectors


In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_VKN9t_D44vRHqfN6mmRtYYW85XgnaB2SPkx7k3vh2ZVY65tV2RDQgRMnZTpfnvRE3EHap")
index = pc.Index("credit-cards")

# Test query
results = index.query(vector=[0.1]*384, top_k=3)
print(results)

QueryResponse(matches=[ScoredVector(id='HDFC_Infinia_chunk_39', score=0.0152001614, values=[]), ScoredVector(id='HDFC_Regalia_chunk_17', score=0.00975992065, values=[]), ScoredVector(id='HDFC_Regalia_chunk_12', score=0.00555265974, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Wed, 01 Jul 2026 11:17:35 GMT', 'content-type': 'application/json', 'content-length': '259', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '116', 'x-envoy-upstream-service-time': '116', 'x-pinecone-response-duration-ms': '118', 'grpc-status': '0', 'server': 'envoy'}))


## Using Faiss as local because pineconde i am getting certficate issue in my laptop

In [ ]:
import json
import numpy as np
!pip install faiss-cpu
import faiss
from sentence_transformers import SentenceTransformer

# Load your chunks
with open('chunks.json', 'r') as f:
    chunks = json.load(f)

# Generate embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c['chunk_text'] for c in chunks]
embeddings = model.encode(texts)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

# Save index
faiss.write_index(index, 'faiss_index.bin')

# Save chunks mapping
with open('chunks_mapping.json', 'w') as f:
    json.dump(chunks, f)

# Download both files
from google.colab import files
files.download('faiss_index.bin')
files.download('chunks_mapping.json')
print("✓ FAISS index + chunks downloaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 27.7 MB/s eta 0:00:00


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ FAISS index + chunks downloaded


In [ ]:
import torch
import shutil

# After generating FAISS, download and save the model
model_path = 'all-MiniLM-L6-v2'
model.save(model_path)

# Download it
!zip -r all-MiniLM-L6-v2.zip all-MiniLM-L6-v2/
files.download('all-MiniLM-L6-v2.zip')

print("✓ Model downloaded")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: all-MiniLM-L6-v2/ (stored 0%)
  adding: all-MiniLM-L6-v2/tokenizer_config.json (deflated 51%)
  adding: all-MiniLM-L6-v2/modules.json (deflated 64%)
  adding: all-MiniLM-L6-v2/config.json (deflated 52%)
  adding: all-MiniLM-L6-v2/1_Pooling/ (stored 0%)
  adding: all-MiniLM-L6-v2/1_Pooling/config.json (deflated 16%)
  adding: all-MiniLM-L6-v2/README.md (deflated 64%)
  adding: all-MiniLM-L6-v2/model.safetensors (deflated 9%)
  adding: all-MiniLM-L6-v2/config_sentence_transformers.json (deflated 40%)
  adding: all-MiniLM-L6-v2/sentence_bert_config.json (deflated 43%)
  adding: all-MiniLM-L6-v2/tokenizer.json (deflated 71%)
  adding: all-MiniLM-L6-v2/2_Normalize/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Model downloaded
